# Fine-Tuned Basic Pitch — Real-Pipeline Evaluation (Quick Version)

Measures whether the fine-tuning gain survives in Basic Pitch's **real sliding-window inference** (what your `predict()` pipeline uses), not the non-overlapping 2s chunks the fine-tune eval used.

**Method (leakage-free):**
- Both pretrained + fine-tuned models go through the **same** overlapping-window inference path.
- Thresholds tuned on **player 04 (validation)**.
- Reported on **player 05 (test)** — the held-out player the fine-tuned model never trained on.
- Fine-tuned model trained on players 00–04, so player 05 is the only honest test set.

**Cells:** (1) env+patches → (2) drive+data → (3) load both models → (4) windowed inference → (5) tune thresholds on val → (6) report on test.


## 1. Environment + Basic Pitch patches (clean from fresh runtime)

In [1]:
!nvidia-smi -L

# Install Basic Pitch
import os
if not os.path.exists('/content/basic-pitch'):
    !git clone -q https://github.com/spotify/basic-pitch.git /content/basic-pitch
!pip install -q --no-deps -e /content/basic-pitch 2>/dev/null
!pip install -q librosa soundfile sox mirdata tensorflow mir_eval resampy==0.4.2
!apt-get install -y sox > /dev/null 2>&1

import sys
sys.path.insert(0, '/content/basic-pitch')
import warnings; warnings.filterwarnings('ignore')
import tensorflow as tf
print(f"TF: {tf.__version__}")

# --- TF 2.20 / Keras 3 compatibility patches (same as fine-tune notebook) ---
# Patch 1: signal.py  input_shape.rank -> len(input_shape)
p = '/content/basic-pitch/basic_pitch/layers/signal.py'
s = open(p).read().replace('rank = input_shape.rank', 'rank = len(input_shape)')
open(p,'w').write(s)

# Patch 2: models.py  raw tf.expand_dims -> Lambda-wrapped
p = '/content/basic-pitch/basic_pitch/models.py'
s = open(p).read()
s = s.replace('x = tf.expand_dims(x, -1)\n    if use_batchnorm:',
              'x = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x)\n    if use_batchnorm:')
s = s.replace('x_contours_reduced = tf.expand_dims(x_contours, -1)',
              'x_contours_reduced = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x_contours)')
open(p,'w').write(s)

# Patch 3: nn.py  remove tf.debugging.assert_equal
p = '/content/basic-pitch/basic_pitch/nn.py'
s = open(p).read().replace('tf.debugging.assert_equal(tf.shape(x).shape, 4)',
                           'pass  # removed for TF2.20/Keras3')
open(p,'w').write(s)

import importlib
import basic_pitch.layers.signal as _sig; importlib.reload(_sig)
import basic_pitch.nn as _nn; importlib.reload(_nn)
import basic_pitch.models as _m; importlib.reload(_m)
import basic_pitch.models as bp_models
print("Patches applied, basic_pitch ready.")

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-dcfcd928-b4b1-0435-fa9d-1cf8a298d787)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basic-pitch (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 136.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.8/263.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━

TF: 2.20.0
Patches applied, basic_pitch ready.


## 2. Mount Drive + locate data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import glob, shutil
from pathlib import Path

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
]
DATA_ROOT = next((c for c in DATA_ROOT_CANDIDATES if (c/'JamsFiles').exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("GuitarSet not found")

LOCAL_AUDIO, LOCAL_JAMS = '/content/gs_audio', '/content/gs_jams'
os.makedirs(LOCAL_AUDIO, exist_ok=True); os.makedirs(LOCAL_JAMS, exist_ok=True)

def copy_if_needed(src_dir, dst_dir, ext):
    src = glob.glob(os.path.join(str(src_dir), f'*.{ext}'))
    if len(glob.glob(os.path.join(dst_dir, f'*.{ext}'))) >= len(src):
        print(f"  {ext}: already on SSD"); return
    print(f"  copying {len(src)} {ext}...")
    for f in src:
        try: shutil.copy2(f, dst_dir)
        except shutil.SameFileError: pass

copy_if_needed(DATA_ROOT/'AudioFiles', LOCAL_AUDIO, 'wav')
copy_if_needed(DATA_ROOT/'JamsFiles',  LOCAL_JAMS,  'jams')

FT_CKPT = '/content/drive/MyDrive/Capstone/outputs/bp_finetune_v2/best_model.keras'
print(f"Audio: {len(glob.glob(LOCAL_AUDIO+'/*.wav'))} | JAMS: {len(glob.glob(LOCAL_JAMS+'/*.jams'))}")
print(f"Fine-tuned checkpoint exists: {os.path.exists(FT_CKPT)}")

Mounted at /content/drive
  copying 360 wav...
  copying 360 jams...
Audio: 360 | JAMS: 360
Fine-tuned checkpoint exists: True


## 3. Load both models — pretrained (baseline) + fine-tuned

In [3]:
import numpy as np

# --- Pretrained Basic Pitch (SavedModel) ---
from basic_pitch import ICASSP_2022_MODEL_PATH
pretrained = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
print("Pretrained SavedModel loaded.")

# --- Fine-tuned model: rebuild architecture, load weights ---
ft_model = bp_models.model()
try:
    ft_model.load_weights(FT_CKPT)
    print("Fine-tuned weights loaded via load_weights().")
except Exception as e:
    print(f"load_weights failed ({type(e).__name__}), trying full load_model...")
    def weighted_bce(y_true, y_pred):
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, label_smoothing=0.05)
        w = 1.0 + 4.0 * tf.reduce_mean(y_true, axis=-1)
        return tf.reduce_mean(w * bce)
    from basic_pitch import nn as bpnn
    from basic_pitch.layers import signal as bpsig, nnaudio as bpna
    custom = {'weighted_bce': weighted_bce}
    for nm in ['FlattenAudioCh','FlattenFreqCh','HarmonicStacking']:
        if hasattr(bpnn,nm): custom[nm]=getattr(bpnn,nm)
    if hasattr(bpsig,'NormalizedLog'): custom['NormalizedLog']=bpsig.NormalizedLog
    for nm in ['CQT','CQT2010v2']:
        if hasattr(bpna,nm): custom[nm]=getattr(bpna,nm)
    ft_model = tf.keras.models.load_model(FT_CKPT, custom_objects=custom, compile=False)
    print("Fine-tuned model loaded via load_model().")

# Sanity: fine-tuned should output LOWER confidence than pretrained (label smoothing)
import librosa
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES
_test = sorted(glob.glob(LOCAL_AUDIO+'/05_*mic*.wav'))[0]
_y,_ = librosa.load(_test, sr=AUDIO_SAMPLE_RATE, mono=True)
_x = tf.constant(_y[:AUDIO_N_SAMPLES].reshape(1,-1,1), tf.float32)
_op = pretrained.signatures['serving_default'](input_2=_x)['onset'].numpy().max()
_of = ft_model(_x, training=False)['onset'].numpy().max()
print(f"onset max — pretrained: {_op:.3f} | fine-tuned: {_of:.3f}")

Pretrained SavedModel loaded.
Fine-tuned weights loaded via load_weights().
onset max — pretrained: 0.930 | fine-tuned: 0.404


## 4. Basic Pitch sliding-window inference
Replicates `predict()`'s overlapping-window inference: 2s windows, 30-frame overlap, trim 15 frames per side on unwrap. Applied **identically** to both models, so the delta is fair.

In [4]:
import json
from basic_pitch.constants import FFT_HOP, ANNOTATIONS_FPS
from basic_pitch.note_creation import model_output_to_notes

N_OVERLAPPING_FRAMES = 30
OVERLAP_LEN = N_OVERLAPPING_FRAMES * FFT_HOP        # 7680
HOP_SIZE    = AUDIO_N_SAMPLES - OVERLAP_LEN          # 36164

def window_audio(audio):
    audio = np.concatenate([np.zeros(OVERLAP_LEN//2, np.float32), audio.astype(np.float32)])
    out = []
    start = 0
    while start < len(audio):
        w = audio[start:start+AUDIO_N_SAMPLES]
        if len(w) < AUDIO_N_SAMPLES:
            w = np.pad(w, (0, AUDIO_N_SAMPLES-len(w)))
        out.append(w)
        if start + AUDIO_N_SAMPLES >= len(audio): break
        start += HOP_SIZE
    return np.stack(out)

def unwrap(stacked, orig_len):
    n = N_OVERLAPPING_FRAMES // 2                    # 15
    trimmed = stacked[:, n:-n, :]
    flat = trimmed.reshape(-1, trimmed.shape[-1])
    n_frames = int(np.floor(orig_len * (ANNOTATIONS_FPS / AUDIO_SAMPLE_RATE)))
    return flat[:n_frames, :]

def infer(model, audio_path, is_saved):
    y,_ = librosa.load(audio_path, sr=AUDIO_SAMPLE_RATE, mono=True)
    windows = window_audio(y)                        # (nw, S)
    x = tf.constant(windows[..., None], tf.float32)  # (nw, S, 1)
    out = (model.signatures['serving_default'](input_2=x) if is_saved
           else model(x, training=False))
    return {'onset':  unwrap(out['onset'].numpy(),  len(y)),
            'note':   unwrap(out['note'].numpy(),   len(y)),
            'contour':unwrap(out['contour'].numpy(),len(y))}

def load_gt(jp):
    jam = json.load(open(jp)); notes=[]
    for ann in jam.get('annotations',[]):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for o in ann['data']:
            notes.append({'onset':float(o['time']),
                          'offset':float(o['time'])+float(o['duration']),
                          'midi':int(round(float(o['value'])))})
    return sorted(notes, key=lambda n:n['onset'])

def match(gt, pred, tol=0.05):
    cands=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p['midi'])!=int(g['midi']): continue
            if abs(p['onset']-g['onset'])<=tol: cands.append((abs(p['onset']-g['onset']),pi,gi))
    cands.sort(); up,ug=set(),set()
    for _,pi,gi in cands:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0; R=tp/(tp+fn) if tp+fn else 0
    return P,R,(2*P*R/(P+R) if P+R else 0)

print("Inference helpers ready.")

Inference helpers ready.


## 5. Cache outputs + tune thresholds on player 04 (validation)

In [5]:
import pandas as pd

def cache_player(model, is_saved, player, label):
    jams = [j for j in sorted(glob.glob(LOCAL_JAMS+'/*.jams'))
            if os.path.basename(j).split('_')[0]==player]
    out={}
    print(f"Caching {label} on player {player} ({len(jams)} recs)...")
    for i,jp in enumerate(jams):
        stem=os.path.splitext(os.path.basename(jp))[0]
        cands=(glob.glob(os.path.join(LOCAL_AUDIO,stem+'*mic*.wav')) or
               glob.glob(os.path.join(LOCAL_AUDIO,stem+'*.wav')))
        if not cands: continue
        out[stem]={**infer(model,cands[0],is_saved),'gt':load_gt(jp)}
        if (i+1)%15==0: print(f"  {i+1}/{len(jams)}")
    return out

MIN_NOTE_LEN = int(np.round(58/1000*ANNOTATIONS_FPS))
def eval_cache(cache, ot, ft):
    rows=[]
    for d in cache.values():
        _,ev = model_output_to_notes({'onset':d['onset'],'note':d['note'],'contour':d['contour']},
                  onset_thresh=ot, frame_thresh=ft, min_note_len=MIN_NOTE_LEN,
                  min_freq=None, max_freq=None, include_pitch_bends=False)
        pred=[{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in ev]
        P,R,F=match(d['gt'],pred); rows.append({'P50':P,'R50':R,'F50':F,'n':len(d['gt'])})
    df=pd.DataFrame(rows)
    return {k:float(np.average(df[k],weights=df.n)) for k in ['P50','R50','F50']}

def best_threshold(cache, onsets, frames):
    best=None
    for ot in onsets:
        for ft in frames:
            a=eval_cache(cache,ot,ft)
            if best is None or a['F50']>best['F50']: best={**a,'onset_t':ot,'frame_t':ft}
    return best

# Cache validation (player 04) for both models
val_pre = cache_player(pretrained, True,  '04', 'pretrained')
val_ft  = cache_player(ft_model,   False, '04', 'fine-tuned')

ONSETS=[0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]
FRAMES=[0.10,0.20,0.30]
best_pre = best_threshold(val_pre, ONSETS, FRAMES)
best_ft  = best_threshold(val_ft,  ONSETS, FRAMES)
print(f"\nBest VAL threshold — pretrained: onset={best_pre['onset_t']} frame={best_pre['frame_t']} (val F50={best_pre['F50']:.4f})")
print(f"Best VAL threshold — fine-tuned: onset={best_ft['onset_t']} frame={best_ft['frame_t']} (val F50={best_ft['F50']:.4f})")

Caching pretrained on player 04 (60 recs)...
  15/60
  30/60
  45/60
  60/60
Caching fine-tuned on player 04 (60 recs)...
  15/60
  30/60
  45/60
  60/60

Best VAL threshold — pretrained: onset=0.5 frame=0.3 (val F50=0.7170)
Best VAL threshold — fine-tuned: onset=0.1 frame=0.3 (val F50=0.7290)


## 6. Report on player 05 (test) — each model at its best validation threshold

In [6]:
# Cache test (player 05) for both models
test_pre = cache_player(pretrained, True,  '05', 'pretrained')
test_ft  = cache_player(ft_model,   False, '05', 'fine-tuned')

res_pre = eval_cache(test_pre, best_pre['onset_t'], best_pre['frame_t'])
res_ft  = eval_cache(test_ft,  best_ft['onset_t'],  best_ft['frame_t'])

print("\n" + "="*64)
print("REAL-PIPELINE EVAL (overlapping-window inference, player 05 held out)")
print("="*64)
print(f"{'':28s} {'onset':>6} {'frame':>6} {'P50':>7} {'R50':>7} {'F50':>7}")
print(f"{'Pretrained Basic Pitch':28s} {best_pre['onset_t']:>6} {best_pre['frame_t']:>6} {res_pre['P50']:>7.4f} {res_pre['R50']:>7.4f} {res_pre['F50']:>7.4f}")
print(f"{'Fine-tuned Basic Pitch':28s} {best_ft['onset_t']:>6} {best_ft['frame_t']:>6} {res_ft['P50']:>7.4f} {res_ft['R50']:>7.4f} {res_ft['F50']:>7.4f}")
print(f"{'Delta':28s} {'':>6} {'':>6} {res_ft['P50']-res_pre['P50']:>+7.4f} {res_ft['R50']-res_pre['R50']:>+7.4f} {res_ft['F50']-res_pre['F50']:>+7.4f}")
print("="*64)
print("Thresholds tuned on player 04, reported on player 05. Both models same inference path.")

Caching pretrained on player 05 (60 recs)...
  15/60
  30/60
  45/60
  60/60
Caching fine-tuned on player 05 (60 recs)...
  15/60
  30/60
  45/60
  60/60

REAL-PIPELINE EVAL (overlapping-window inference, player 05 held out)
                              onset  frame     P50     R50     F50
Pretrained Basic Pitch          0.5    0.3  0.6959  0.8989  0.7704
Fine-tuned Basic Pitch          0.1    0.3  0.7550  0.8629  0.7946
Delta                                      +0.0591 -0.0360 +0.0242
Thresholds tuned on player 04, reported on player 05. Both models same inference path.
